# Amazon Appliance Insights — Product metadata RDD
Task 2 | Databricks, Cluster 1, Aditya Verma

Loads all of the products metadata from GCS as a spark RDD, while checking quality, keeping needed fields for the later joining, and saving the cleaned data as validated JSONL.

Run after `01_create_reviews_rdd`. Reuses the group GCP secret.

### 1. Connect to GCP and load the raw dataset

In [0]:
# Read GCP credentials from Databricks secrets
gcp_client_email = dbutils.secrets.get(scope="gcp_scope", key="client_email")
gcp_project_id = dbutils.secrets.get(scope="gcp_scope", key="project_id")
gcp_private_key = dbutils.secrets.get(scope="gcp_scope", key="private_key")
gcp_private_key_id = dbutils.secrets.get(scope="gcp_scope", key="private_key_id")

# Configure Spark to access GCS
sc = spark.sparkContext
hconf = sc._jsc.hadoopConfiguration()
hconf.set("google.cloud.auth.service.account.enable", "true")
hconf.set("fs.gs.project.id", gcp_project_id)
hconf.set("fs.gs.auth.service.account.email", gcp_client_email)
hconf.set("fs.gs.auth.service.account.private.key", gcp_private_key)
hconf.set("fs.gs.auth.service.account.private.key.id", gcp_private_key_id)

metadata_path = "gs://amazon_appliance/raw/meta_Appliances.jsonl"
raw_meta_rdd = sc.textFile(metadata_path)
print("GCP access successful:", bool(raw_meta_rdd.take(1)))

### 2. Create the RDD and inspect fields
If a line is malformed or is not a JSON object, count it separately.

In [0]:
import json

def parse_json(x):
    try:
        x = json.loads(x)
        if isinstance(x, dict):
            return x
    except ValueError:
        pass
    return None

parsed_meta_rdd = raw_meta_rdd.map(parse_json).cache()
meta_rdd = parsed_meta_rdd.filter(lambda x: x is not None)

total = raw_meta_rdd.count()
valid = meta_rdd.count()
print("total metadata records:", total)
print("malformed records:", total - valid)
print("fields:", list(meta_rdd.first().keys()))

### 3. Check missing values and data types
The expected product rating field is `average_rating`. If the actual dataset uses `rating`, use that instead. Keep product ratings separate from individual review ratings when joining.

Findings: `price` is missing for 47,601 of 94,327 products. Prices are are either a number or null with no NaN. Range was set from $0.01 to $21,095.62, kept this way as recorded. Ratings are always between 1.0 star and 5.0 star, `parent_asin` is a 10 character string with no extra spaces.

In [0]:
fields = list(meta_rdd.first().keys())

if "average_rating" in fields:
    rating_field = "average_rating"
else:
    rating_field = "rating"

keep_fields = ["parent_asin", "title", "store", "price", rating_field]
assert all(x in fields for x in keep_fields), "Check the actual metadata fields before continuing"

for field in keep_fields:
    missing = meta_rdd.filter(lambda x: x.get(field) is None or x.get(field) == "").count()
    print(field, "missing:", missing)

print("price types:", meta_rdd.map(lambda x: type(x.get("price")).__name__).distinct().collect())
print("rating types:", meta_rdd.map(lambda x: type(x.get(rating_field)).__name__).distinct().collect())
print("invalid join keys:", meta_rdd.filter(lambda x: not isinstance(x.get("parent_asin"), str) or x.get("parent_asin", "").strip() == "").count())

In [0]:
import math

# price: look only at products that have a price
prices = meta_rdd.map(lambda x: x.get("price")).filter(lambda x: x is not None)
print("products with a price:", prices.count())
print("price NaN:", prices.filter(lambda x: math.isnan(x)).count())
print("price 0 or below:", prices.filter(lambda x: x <= 0).count())
print("lowest price:", prices.min())
print("highest price:", prices.max())

# product rating should be between 1 and 5
ratings = meta_rdd.map(lambda x: x["average_rating"])
print("rating NaN:", ratings.filter(lambda x: math.isnan(x)).count())
print("rating outside 1-5:", ratings.filter(lambda x: x < 1 or x > 5).count())
print("lowest rating:", ratings.min())
print("highest rating:", ratings.max())

# is a missing title/store stored as null or as empty text?
print("title null:", meta_rdd.filter(lambda x: x.get("title") is None).count())
print("title empty text:", meta_rdd.filter(lambda x: x.get("title") == "").count())
print("store null:", meta_rdd.filter(lambda x: x.get("store") is None).count())
print("store empty text:", meta_rdd.filter(lambda x: x.get("store") == "").count())

# parent_asin: Amazon IDs are normally 10 letters/digits with no spaces
print("parent_asin not 10 characters:", meta_rdd.filter(lambda x: len(x["parent_asin"]) != 10).count())
print("parent_asin with extra spaces:", meta_rdd.filter(lambda x: x["parent_asin"] != x["parent_asin"].strip()).count())
print("parent_asin examples:", meta_rdd.map(lambda x: x["parent_asin"]).take(5))

### 4. Check and remove exact duplicate records
This compares raw JSON lines, like the simplified reviews notebook. Does not remove different records that happen to share a product ID.

Checks for duplicates with `distinct()`. Does the same for records with the `parent_asin` with `reduceByKey()`. 

Result: 0 exact duplicates and 0 duplicate product IDs, so every product already has exactly one record.

In [0]:
# exact duplicates: the same line appears more than once
unique_raw_rdd = raw_meta_rdd.distinct()
unique_meta_rdd = unique_raw_rdd.map(parse_json).filter(lambda x: x is not None)
unique = unique_meta_rdd.count()
print("valid metadata records:", valid)
print("unique metadata records:", unique)
print("exact duplicate records:", valid - unique)

# duplicate product IDs: different lines that share the same parent_asin
dup_ids = unique_meta_rdd.map(lambda x: (x["parent_asin"], 1)).reduceByKey(lambda x, y: x + y).filter(lambda x: x[1] > 1)
print("duplicate product IDs:", dup_ids.count())
print("examples:", dup_ids.take(5))

### 5. Clean product metadata
Keeps the five fields needed for the join: `parent_asin`, `title`, `store`, `price`, `average_rating`.
- No products are removed. No malformed lines, invalid join keys or duplicates are present
- Missing prices stay null, not replaced with 0, products not dropped.
- Every missing value is stored as null, 9 empty titles
- No number conversion is needed, we already have clean numbers

In [0]:
# keep only the fields needed for the join
def clean_meta(x):
    return {
        "parent_asin": x["parent_asin"],
        "title": x.get("title") or None,   # empty title "" becomes None (null)
        "store": x.get("store"),           # missing stores are already None
        "price": x.get("price"),           # keep None, never replace with 0
        "average_rating": x["average_rating"]
    }

cleaned_meta_rdd = unique_meta_rdd.map(clean_meta).cache()

cleaned_count = cleaned_meta_rdd.count()
print("cleaned metadata records:", cleaned_count)
print("fields:", list(cleaned_meta_rdd.first().keys()))
print("missing title:", cleaned_meta_rdd.filter(lambda x: x["title"] is None).count())
print("missing store:", cleaned_meta_rdd.filter(lambda x: x["store"] is None).count())
print("missing price:", cleaned_meta_rdd.filter(lambda x: x["price"] is None).count())
print("missing rating:", cleaned_meta_rdd.filter(lambda x: x["average_rating"] is None).count())
print("sample:", cleaned_meta_rdd.take(2))

### 6. Check uniqueness of the join key
The proposal expects one metadata record per product and many reviews per product. If duplicate product IDs remain, inspect the conflicting records and agree on a rule before saving. Do not randomly pick a record.

In [0]:
duplicate_keys = cleaned_meta_rdd.map(lambda x: (x["parent_asin"], 1)).reduceByKey(lambda x, y: x + y).filter(lambda x: x[1] > 1)

print("duplicate product IDs:", duplicate_keys.count())
print("sample duplicate IDs:", duplicate_keys.take(5))
assert duplicate_keys.isEmpty(), "Resolve duplicate parent_asin values before saving"

### 7. Save the cleaned RDD to GCP
Spark writes a directory containing part files. The Hadoop filesystem check uses the same GCP configuration as Spark. Do not overwrite existing output automatically. If the cleaning changes, coordinate a deliberate regeneration before the join.

In [0]:
output_path = "gs://amazon_appliance/processed/meta_clean/"

assert duplicate_keys.isEmpty(), "Resolve duplicate product IDs first"
path = sc._jvm.org.apache.hadoop.fs.Path(output_path)
fs = path.getFileSystem(hconf)

if fs.exists(path):
    print("Output already exists. Checking saved data in the next cell.")
else:
    cleaned_meta_rdd.map(json.dumps).saveAsTextFile(output_path)
    print("cleaned metadata saved successfully")

### 8. Read back and validate the saved data
Confirm record counts, schema and unique product IDs. If an old output exists, compare its records too before trusting it.

In [0]:
saved_meta_rdd = sc.textFile(output_path).map(json.loads)
saved_count = saved_meta_rdd.count()
saved_keys = saved_meta_rdd.map(lambda x: x["parent_asin"])
missing_keys = saved_keys.filter(lambda x: not isinstance(x, str) or x.strip() == "").count()
unique_keys = saved_keys.distinct().count()

print("saved metadata records:", saved_count)
print("fields:", list(saved_meta_rdd.first().keys()))
print("invalid join keys:", missing_keys)
print("unique product IDs:", unique_keys)

assert saved_count == cleaned_count
assert missing_keys == 0
assert unique_keys == saved_count

# Compare saved records with the current cleaned RDD
expected = cleaned_meta_rdd.map(lambda x: (x["parent_asin"], x))
saved = saved_meta_rdd.map(lambda x: (x["parent_asin"], x))
differences = expected.join(saved).filter(lambda x: x[1][0] != x[1][1]).count()
assert expected.keys().subtract(saved.keys()).isEmpty()
assert differences == 0, "Saved output differs from the current cleaned data"
print("validation successful")

In [0]:
# look at one saved line exactly as it is stored (JSONL, missing values as null)
print(sc.textFile(output_path).take(1))

# compare product IDs with the cleaned reviews (Jagriti's output)
reviews_rdd = sc.textFile("gs://amazon_appliance/processed/reviews_clean/").map(json.loads)
review_keys = reviews_rdd.map(lambda x: x["parent_asin"]).distinct()
meta_keys = saved_meta_rdd.map(lambda x: x["parent_asin"])

print("reviews:", reviews_rdd.count())
print("review parent_asin types:", reviews_rdd.map(lambda x: type(x["parent_asin"]).__name__).distinct().collect())
print("reviewed products:", review_keys.count())
print("reviewed products with metadata:", review_keys.intersection(meta_keys).count())
print("reviewed products without metadata:", review_keys.subtract(meta_keys).count())